# Customer Segmentation with K-Means

This notebook performs an end-to-end unsupervised machine learning workflow for customer segmentation.

The objective is to clean customer-level data, engineer useful behavioural features, select relevant variables, identify customer groups with K-Means clustering, and interpret the resulting clusters from a business perspective.

## 1. Imports and project paths

The notebook uses relative paths, so it can run from the GitHub repository without Google Drive or Colab-specific folders.

Expected structure:

```text
customer-segmentation-kmeans/
├── customer_segmentation_kmeans.ipynb
├── data/
│   └── input/
│       └── raw_database.csv
├── reports/
│   ├── raw_database_report.html
│   └── modified_database_report.html
└── outputs/
```

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "input"
REPORTS_DIR = PROJECT_ROOT / "reports"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
OUTPUTS_DIR.mkdir(exist_ok=True, parents=True)

raw_database_path = DATA_DIR / "raw_database.csv"

print(f"Reading data from: {raw_database_path}")

## 2. Data loading

In [ ]:
db_raw = pd.read_csv(raw_database_path)

print(f"Rows: {db_raw.shape[0]:,}")
print(f"Columns: {db_raw.shape[1]}")
db_raw.head()

## 3. Initial data understanding

The dataset contains customer-level information such as supply type, payment behaviour, number of reminders, suspension requests, tenure, contracts and average bill/payment-related variables.

In [ ]:
db_raw.info()

In [ ]:
missing_summary = (
    db_raw.isna()
    .sum()
    .to_frame("missing_values")
    .assign(missing_pct=lambda x: 100 * x["missing_values"] / len(db_raw))
    .sort_values("missing_values", ascending=False)
)

missing_summary

In [ ]:
db_raw.describe(include="all").T

## 4. Data cleaning

The cleaning steps are designed to make the dataset suitable for clustering:

- missing suspension requests are interpreted as zero suspension requests;
- the euro symbol is removed from `avg_amount`;
- monetary and numerical columns are converted to numeric values;
- negative values in naturally non-negative fields are replaced with zero;
- the categorical `supply` variable is one-hot encoded.

In [ ]:
db_clean = db_raw.copy()

# Missing values: if no suspension request is recorded, treat it as zero
db_clean["num_suspension_request"] = db_clean["num_suspension_request"].fillna(0).astype(int)

# Convert amount from strings such as "100.0€" to numeric values
db_clean["avg_amount"] = (
    db_clean["avg_amount"]
    .astype(str)
    .str.replace("€", "", regex=False)
    .str.replace(",", ".", regex=False)
)

db_clean["avg_amount"] = pd.to_numeric(db_clean["avg_amount"], errors="coerce").fillna(0)

# Replace negative values in fields that should be non-negative
non_negative_cols = [
    "tenure_years",
    "num_contracts",
    "num_reminder",
    "num_reminder_soft",
    "num_reminder_hard",
    "num_suspension_request",
    "avg_issue_payment_days",
    "avg_amount",
]

for col in non_negative_cols:
    db_clean[col] = db_clean[col].clip(lower=0)

# One-hot encode supply type
db_clean = pd.get_dummies(db_clean, columns=["supply"], prefix="supply", dtype=int)

db_clean.head()

## 5. Feature engineering

`num_reminders_year` normalises the number of reminders by customer tenure. This makes customers with different relationship lengths more comparable.

In [ ]:
db_clean["num_reminders_year"] = (
    db_clean["num_reminder_soft"] + db_clean["num_reminder_hard"]
) / (db_clean["tenure_years"] + 1)

# num_reminder is the sum of soft and hard reminders, so it is removed to avoid duplication.
db_model = db_clean.drop(columns=["num_reminder"])

db_model.head()

## 6. Feature selection and scaling

K-Means is distance-based, so variables must be standardised before clustering.

In [ ]:
features = [
    "num_contracts",
    "num_suspension_request",
    "avg_issue_payment_days",
    "tenure_years",
    "num_reminders_year",
    "avg_amount",
    "direct_debit",
    "domestic",
    "supply_DUAL",
    "supply_GAS",
    "supply_POWER",
]

X = db_model[features].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X.head()

## 7. Model selection: elbow method and silhouette score

To keep the notebook efficient on GitHub, the model-selection step uses a reproducible sample of the dataset. The final clustering model is then fitted on the full dataset.

In [ ]:
sample_size = min(50000, len(X_scaled))
rng = np.random.default_rng(RANDOM_STATE)
sample_idx = rng.choice(len(X_scaled), size=sample_size, replace=False)
X_sample = X_scaled[sample_idx]

k_values = range(2, 9)
inertias = []
silhouette_scores = []

for k in k_values:
    model = MiniBatchKMeans(
        n_clusters=k,
        random_state=RANDOM_STATE,
        n_init=10,
        batch_size=4096
    )
    labels_sample = model.fit_predict(X_sample)
    inertias.append(model.inertia_)
    silhouette_scores.append(
        silhouette_score(X_sample, labels_sample, sample_size=min(10000, sample_size), random_state=RANDOM_STATE)
    )

model_selection = pd.DataFrame({
    "k": list(k_values),
    "inertia": inertias,
    "silhouette_score": silhouette_scores
})

model_selection

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(model_selection["k"], model_selection["inertia"], marker="o")
plt.xlabel("Number of clusters")
plt.ylabel("Inertia")
plt.title("Elbow Method")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(model_selection["k"], model_selection["silhouette_score"], marker="o")
plt.xlabel("Number of clusters")
plt.ylabel("Silhouette score")
plt.title("Silhouette Score by Number of Clusters")
plt.show()

## 8. Final K-Means model

The final number of clusters is set to 4. This keeps the segmentation interpretable while still separating customers by relevant behavioural patterns.

You can change `FINAL_K` after reviewing the elbow and silhouette charts.

In [ ]:
FINAL_K = 4

final_model = MiniBatchKMeans(
    n_clusters=FINAL_K,
    random_state=RANDOM_STATE,
    n_init=10,
    batch_size=4096
)

db_final = db_model.copy()
db_final["cluster"] = final_model.fit_predict(X_scaled)

db_final[["customer_id", "cluster"]].head()

## 9. Cluster profiling

The table below shows the average characteristics of each cluster. This is the main step for translating the mathematical clustering output into business insights.

In [ ]:
cluster_profile = (
    db_final
    .drop(columns=["customer_id"])
    .groupby("cluster")
    .mean()
    .round(2)
)

cluster_size = (
    db_final["cluster"]
    .value_counts(normalize=False)
    .sort_index()
    .to_frame("customers")
)

cluster_share = (
    db_final["cluster"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
    .to_frame("share_pct")
)

cluster_summary = cluster_size.join(cluster_share).join(cluster_profile)
cluster_summary

In [ ]:
key_profile_cols = [
    "customers", "share_pct", "num_contracts", "num_suspension_request",
    "avg_issue_payment_days", "tenure_years", "num_reminders_year",
    "avg_amount", "direct_debit", "domestic"
]

cluster_summary[key_profile_cols]

## 10. Business interpretation

The following function creates a simple rule-based interpretation of each cluster. It is not meant to replace business judgement, but it helps make the segmentation readable.

In [ ]:
def describe_cluster(row):
    notes = []

    if row["num_suspension_request"] >= cluster_summary["num_suspension_request"].quantile(0.75):
        notes.append("high suspension-request behaviour")
    elif row["num_suspension_request"] <= cluster_summary["num_suspension_request"].quantile(0.25):
        notes.append("low suspension-request behaviour")

    if row["num_reminders_year"] >= cluster_summary["num_reminders_year"].quantile(0.75):
        notes.append("frequent reminders per year")
    elif row["num_reminders_year"] <= cluster_summary["num_reminders_year"].quantile(0.25):
        notes.append("limited reminder activity")

    if row["avg_issue_payment_days"] >= cluster_summary["avg_issue_payment_days"].quantile(0.75):
        notes.append("slower payment behaviour")
    elif row["avg_issue_payment_days"] <= cluster_summary["avg_issue_payment_days"].quantile(0.25):
        notes.append("faster payment behaviour")

    if row["avg_amount"] >= cluster_summary["avg_amount"].quantile(0.75):
        notes.append("higher average bill amount")
    elif row["avg_amount"] <= cluster_summary["avg_amount"].quantile(0.25):
        notes.append("lower average bill amount")

    return "; ".join(notes)

business_interpretation = cluster_summary.copy()
business_interpretation["interpretation"] = business_interpretation.apply(describe_cluster, axis=1)

business_interpretation[["customers", "share_pct", "interpretation"]]

## 11. Export results

The notebook exports:

- a customer-level file with cluster labels;
- a cluster summary table.

In [ ]:
customer_clusters_path = OUTPUTS_DIR / "customer_clusters.csv"
cluster_summary_path = OUTPUTS_DIR / "cluster_summary.csv"

db_final[["customer_id", "cluster"]].to_csv(customer_clusters_path, index=False)
cluster_summary.to_csv(cluster_summary_path)

print(f"Saved customer clusters to: {customer_clusters_path}")
print(f"Saved cluster summary to: {cluster_summary_path}")

## 12. Conclusion

This project provides a complete unsupervised learning workflow for customer segmentation:

1. data loading and inspection;
2. cleaning of missing, categorical and inconsistent values;
3. feature engineering;
4. feature scaling;
5. K-Means clustering;
6. model-selection support through elbow and silhouette analysis;
7. business-oriented cluster interpretation;
8. export of reusable outputs.

The resulting segmentation can support customer monitoring, payment behaviour analysis and targeted business actions.